In [1]:
from datasets import load_dataset

In [2]:
dataset_path="dataset/small_datset.jsonl"
dataset_name = "tssb_data_3M"
dataset = load_dataset("json", data_files=dataset_path, split="train[:80%]")
eval_dataset = load_dataset("json", data_files=dataset_path, split="train[80%:]")

print("dataset: ",dataset)
print("eval_dataset: ",eval_dataset)

dataset:  Dataset({
    features: ['prompt', 'completion'],
    num_rows: 28000
})
eval_dataset:  Dataset({
    features: ['prompt', 'completion'],
    num_rows: 7000
})


In [3]:
def combine_prompt_completion(example):
    return {
        "text": example["prompt"] + " " + example["completion"],
        "buggy_code": example["prompt"],
        "fixed_code": example["completion"]
    }

dataset = dataset.map(combine_prompt_completion)
eval_dataset = eval_dataset.map(combine_prompt_completion)
EVAL_REFERENCES = [ex["fixed_code"] for ex in eval_dataset]
print("dataset:", dataset[0],"\n")
print("eval_dataset:", eval_dataset[0],"\n")
print("eval_references:", EVAL_REFERENCES[0],"\n")

dataset: {'prompt': "##Task: Fix the issues\n##Bug Type: MORE_SPECIFIC_IF\n##Buggy Code:\nif len ( op . metadata [ 'device_id' ] ) == 1 : op . metadata [ 'device_id' ] = '1'\n##Fixed Code:", 'completion': "if 'device_id' in op . metadata and isinstance ( op . metadata [ 'device_id' ] , ( list , tuple ) ) and len ( op . metadata [ 'device_id' ] ) == 1 : op . metadata [ 'device_id' ] = '1'", 'text': "##Task: Fix the issues\n##Bug Type: MORE_SPECIFIC_IF\n##Buggy Code:\nif len ( op . metadata [ 'device_id' ] ) == 1 : op . metadata [ 'device_id' ] = '1'\n##Fixed Code: if 'device_id' in op . metadata and isinstance ( op . metadata [ 'device_id' ] , ( list , tuple ) ) and len ( op . metadata [ 'device_id' ] ) == 1 : op . metadata [ 'device_id' ] = '1'", 'buggy_code': "##Task: Fix the issues\n##Bug Type: MORE_SPECIFIC_IF\n##Buggy Code:\nif len ( op . metadata [ 'device_id' ] ) == 1 : op . metadata [ 'device_id' ] = '1'\n##Fixed Code:", 'fixed_code': "if 'device_id' in op . metadata and isinsta

In [4]:
import torch
cuda_available = torch.cuda.is_available()

if cuda_available:
    device_id = 0  # You can change to 1,2,3 if you want other GPUs
    torch.cuda.set_device(device_id)
    # device = torch.device(f"cuda:{device_id}")
    device = torch.device(f"cuda:{device_id}")
    print(f"🖥️ Using GPU {device_id}: {torch.cuda.get_device_name(device_id)}")
else:
    device = torch.device("cpu")
    print("⚙️ No GPU available, using CPU.")

print(f"Device selected: {device}")

🖥️ Using GPU 0: NVIDIA GeForce RTX 4070 SUPER
Device selected: cuda:0


In [5]:
from unsloth import FastLanguageModel
base_model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = 4096,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.4.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 1. Max memory: 11.994 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2025.4.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [7]:
import evaluate
from codebleu import compute_codebleu
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
acc = evaluate.load("accuracy")


def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    labels = labels[:, 1:]
    preds = preds[:, :-1]

    # Handle padding/masks
    mask = labels == -100
    labels[mask] = tokenizer.pad_token_id
    preds[mask] = tokenizer.pad_token_id
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Extract completions from "##Fixed Code:"
    decoded_completions = []
    for i, pred in enumerate(decoded_preds):
        parts = pred.split("##Fixed Code:")
        completion = parts[-1].strip() if len(parts) > 1 else pred.strip()
        decoded_completions.append(completion)
        # if i < 2:
        #     print(f"[DEBUG] decoded_pred[{i}]:", repr(pred))
        #     print(f"[DEBUG] completion[{i}]:", repr(completion))
        #     print(f"[DEBUG] reference[{i}]:", repr(EVAL_REFERENCES[i]))

    bleu_score = bleu.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    rouge_score = rouge.compute(predictions=decoded_completions, references=EVAL_REFERENCES)
    accuracy = acc.compute(predictions=preds[~mask], references=labels[~mask])

    refs = [[ref] for ref in EVAL_REFERENCES]
    codebleu_scores = compute_codebleu(decoded_completions, refs, lang="python")

    return {
        "codebleu": codebleu_scores["codebleu"],
        **bleu_score,
        **rouge_score,
        **accuracy
    }

In [8]:
import neptune
from transformers.integrations import NeptuneCallback
run = neptune.init_run(
    project="casvi/CodeMedic",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiIzMTMzYjhhOC1jYzA1LTQ0YjAtOTJjNi1iY2EzM2VhMDY0OTcifQ=="
)


[neptune] [warning] NeptuneWarning: By default, these monitoring options are disabled in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', 'capture_hardware_metrics'. You can set them to 'True' when initializing the run and the monitoring will continue until you call run.stop() or the kernel stops. NOTE: To track the source files, pass their paths to the 'source_code' argument. For help, see: https://docs-legacy.neptune.ai/logging/source_code/


[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/casvi/CodeMedic/e/COD-85


In [9]:
from trl import SFTTrainer, SFTConfig
import time

output_dir='./results'
logging_dir='./logs'
# 0.00016215142122972796
learning_rate =1.62e-4
num_epochs =10
batch_size=2
max_steps=1100

start=time.time()
# SFT Config
config = SFTConfig(
    dataset_num_proc = 1,
    output_dir=output_dir,
    logging_dir=logging_dir,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    dataset_text_field="prompt",#Depends on the colum of the data set
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=batch_size,
    num_train_epochs=num_epochs,
    report_to="none",
    save_steps=200,
    logging_steps=200,
    max_steps=max_steps,
    eval_accumulation_steps=100
)

trainer = SFTTrainer(
    model=model,  # base or PEFT model
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    args=config,
    warmup_steps = 5,
    weight_decay = 0.01,
    compute_metrics = compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)
metrics_before_training = trainer.evaluate()
print("Metrics before training:", metrics_before_training)
run["eval/before_training/"] = metrics_before_training

trainer.train()

for record in trainer.state.log_history:
    if "loss" in record:
        step = record.get("step", None)
        loss = record["loss"]
        run["train/loss"].append({"step": step, "value": loss})

metrics_after_training = trainer.evaluate()
print("Metrics after training:", metrics_after_training)
run["eval/after_training/"] = metrics_after_training

run.stop()
end = time.time()
length = end - start

hours = int(length // 3600)
minutes = int((length % 3600) // 60)
seconds = int(length % 60)

print(f"It took {hours} hours, {minutes} minutes, and {seconds} seconds to train the model!")

Unsloth: Tokenizing ["prompt"]:   0%|          | 0/28000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"]:   0%|          | 0/7000 [00:00<?, ? examples/s]

Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Metrics before training: {'eval_loss': 4.072007179260254, 'eval_model_preparation_time': 0.005, 'eval_codebleu': 0.6366504405524039, 'eval_bleu': 0.1113287155445767, 'eval_precisions': [0.2941748156460404, 0.13623485586370704, 0.07841531775524344, 0.048880333516150655], 'eval_brevity_penalty': 1.0, 'eval_length_ratio': 1.7210748670786034, 'eval_translation_length': 299424, 'eval_reference_length': 173975, 'eval_rouge1': 0.1809724847895377, 'eval_rouge2': 0.056187851326925206, 'eval_rougeL': 0.16152859828838664, 'eval_rougeLsum': 0.16453516155112274, 'eval_accuracy': 0.4015099826298368, 'eval_runtime': 164.7684, 'eval_samples_per_second': 42.484, 'eval_steps_per_second': 10.621}


[neptune] [warning] NeptuneUnsupportedType: You're attempting to log a type that is not directly supported by Neptune (<class 'list'>).
        Convert the value to a supported type, such as a string or float, or use stringify_unsupported(obj)
        for dictionaries or collections that contain unsupported values.
        For more, see https://docs-legacy.neptune.ai/help/value_of_unsupported_type
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28,000 | Num Epochs = 1 | Total steps = 1,100
O^O/ \_/ \    Batch size per device = 3 | Gradient accumulation steps = 3
\        /    Data Parallel GPUs = 1 | Total batch size (3 x 3 x 1) = 9
 "-____-"     Trainable parameters = 132,120,576/4,000,000,000 (3.30% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,Model Preparation Time,Codebleu,Bleu,Precisions,Brevity Penalty,Length Ratio,Translation Length,Reference Length,Rouge1,Rouge2,Rougel,Rougelsum,Accuracy
200,1.499500,1.415851,0.005000,0.169179,0.000000,"[0.4311023622047244, 0.2375249500998004, 0.15587044534412955, 0.10882956878850103]",0.000000,0.002920,508,173975,0.000319,0.000081,0.000265,0.000293,0.726993
400,1.380500,1.392284,0.005000,0.149684,0.000000,"[0.4980694980694981, 0.2890625, 0.1976284584980237, 0.138]",0.000000,0.002977,518,173975,0.000259,0.000073,0.000221,0.000244,0.729717
600,1.341200,1.368510,0.005000,0.118577,0.000000,"[0.5698924731182796, 0.33604336043360433, 0.22950819672131148, 0.17079889807162535]",0.000000,0.002138,372,173975,0.000160,0.000066,0.000141,0.000149,0.731996
800,1.275700,1.347734,0.005000,0.210343,0.000000,"[0.5361990950226244, 0.3112128146453089, 0.2222222222222222, 0.17096018735362997]",0.000000,0.002541,442,173975,0.000204,0.000071,0.000187,0.000199,0.736218
1000,1.241800,1.331044,0.005000,0.178887,0.000000,"[0.5091324200913242, 0.2979214780600462, 0.21728971962616822, 0.16784869976359337]",0.000000,0.002518,438,173975,0.000194,0.000075,0.000178,0.000190,0.739305


Metrics after training: {'eval_loss': 1.33104407787323, 'eval_model_preparation_time': 0.005, 'eval_codebleu': 0.1788871658301857, 'eval_bleu': 2.3273438160748247e-173, 'eval_precisions': [0.5091324200913242, 0.2979214780600462, 0.21728971962616822, 0.16784869976359337], 'eval_brevity_penalty': 8.533714595145523e-173, 'eval_length_ratio': 0.0025176031038942376, 'eval_translation_length': 438, 'eval_reference_length': 173975, 'eval_rouge1': 0.00019384672496840778, 'eval_rouge2': 7.514226087858244e-05, 'eval_rougeL': 0.00017813546976889617, 'eval_rougeLsum': 0.00019039052930040518, 'eval_accuracy': 0.7393045379923905, 'eval_runtime': 167.0244, 'eval_samples_per_second': 41.91, 'eval_steps_per_second': 10.478}
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Waiting for the remaining 16 operations to synchronize with Neptune. Do not kill this process.
[neptune] [info   ] All 16 operations synced, thanks for waiting!
[

In [10]:
import os
trainer.model.save_pretrained(os.path.join(output_dir, "final_checkpoint"))
tokenizer.save_pretrained(os.path.join(output_dir, "final_checkpoint"))
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('./results\\final_checkpoint\\tokenizer_config.json',
 './results\\final_checkpoint\\special_tokens_map.json',
 './results\\final_checkpoint\\vocab.json',
 './results\\final_checkpoint\\merges.txt',
 './results\\final_checkpoint\\added_tokens.json',
 './results\\final_checkpoint\\tokenizer.json')

In [11]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from peft import PeftModel
import os
base_model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit"
output_dir='./results'
logging_dir='./logs'

base_model, _ = FastLanguageModel.from_pretrained(
    model_name = base_model_name, # MODEL USED FOR TRAINING
    max_seq_length = 2048,
    load_in_4bit = True,
)

adapter_path = os.path.join(output_dir, "final_checkpoint")
model = PeftModel.from_pretrained(base_model, adapter_path)
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)

messages = [
{"role": "user", "content": """I have a Python function that's supposed to process a list of numbers, but it's not working as expected. Here's my code:

def process_numbers(numbers):
    result = []
    for i in range(len(numbers)):
        if numbers[i] > numbers[i+1]:
            result.append(numbers[i])
    return sum(result)

numbers = [5, 2, 8, 1, 9]
print(process_numbers(numbers))

The code keeps throwing an IndexError. Can you help me fix this and explain what's wrong?"""}
]

# Solution
# def process_numbers(numbers):
#     result = []
#     for i in range(len(numbers) - 1) :
#         if numbers[i] > numbers[i+1] :
#             result . append(numbers[i])
#     return sum(result)

# numbers = [5, 2, 8, 1, 9]
# print(process_numbers(numbers))


text = tokenizer.apply_chat_template(
messages,
tokenize = False,
add_generation_prompt = True,
enable_thinking = False,
)

from transformers import TextStreamer
_ = model.generate(
**tokenizer(text, return_tensors = "pt").to("cuda"),
max_new_tokens = 256, # Increase for longer outputs!
temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
streamer = TextStreamer(tokenizer, skip_prompt = True),
)

==((====))==  Unsloth 2025.4.7: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4070 SUPER. Num GPUs = 1. Max memory: 11.994 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
You're getting an `IndexError` because you're trying to access `numbers[i+1]` when `i` is the last index (i.e., `i == len(numbers) - 1`), which would be out of bounds.

### Fixed Code:
def process_numbers(numbers):
    result = []
    for i in range(len(numbers) - 1):
        if numbers[i] > numbers[i+1]:
            result.append(numbers[i])
    return sum(result)

### Explanation:
- The loop should iterate from `0` to `len(numbers) - 2` (inclusive), because `i+1` must be less than `len(numbers)`.
- This ensures that we o